# Baloot Dataset — Tester Workflow

**Project:** `hakim-vision`, the computer-vision pillar of the Hakim Baloot AI platform.

**Baloot** is a Saudi trick-taking card game played with a **36-card deck** — ranks **A, K, Q, J, 10, 9, 8, 7, 6** in each of the four suits (hearts `h`, diamonds `d`, clubs `c`, spades `s`).

## What this notebook does

This is a **smoke test**, not a full dataset generator. It exercises the synthetic-scene pipeline that lives in `src/hakim_vision/` end-to-end on a handful of scenes so you can verify the pipeline works on your machine before kicking off a real dataset build with the CLI.

**Outcome:** if every cell below shows ✅ and the rendered scenes in Section 5 look like reasonable cards on textured backgrounds, the pipeline is healthy. Section 8 has the tester sign-off checklist.

**Filing findings:** if a cell fails, copy the traceback and the output of `uv run hakim-vision config-show` into a GitHub issue. See [CONTRIBUTING.md](CONTRIBUTING.md).

## 1. Prerequisites

Before running this notebook:

1. `uv sync --all-extras` from the repo root.
2. Pack your asset shards (only needed once per machine):
   ```powershell
   uv run hakim-vision pack-backgrounds <textures_dir> data/shards/backgrounds
   uv run hakim-vision pack-cards       <cards_dir>    data/shards/cards
   ```
   Adjust `BACKGROUNDS_DIR` and `CARDS_DIR` in the cell below if your shards live elsewhere.

The next cell prints versions and asserts the package imports.

In [ ]:
import sys
import cv2
import numpy as np
import hakim_vision

print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {np.__version__}")
print(f"OpenCV      : {cv2.__version__}")
print(f"hakim_vision: {hakim_vision.__version__}")
print("✅ environment ready")

## 2. Deck sanity check

Confirm `data/cards.names` declares exactly the 36 Baloot cards. If this fails, the file has drifted from the Baloot deck spec and downstream label IDs cannot be trusted.

In [ ]:
from pathlib import Path

BALOOT_RANKS = ("A", "K", "Q", "J", "10", "9", "8", "7", "6")
BALOOT_SUITS = ("h", "d", "c", "s")
EXPECTED = {f"{r}{s}" for r in BALOOT_RANKS for s in BALOOT_SUITS}

names_path = Path("data/cards.names")
declared = {line.strip() for line in names_path.read_text(encoding="utf-8").splitlines() if line.strip()}

assert len(declared) == 36, f"expected 36 cards, got {len(declared)}"
assert declared == EXPECTED, (
    f"deck mismatch.\n  unexpected: {sorted(declared - EXPECTED)}\n  missing   : {sorted(EXPECTED - declared)}"
)
print(f"✅ data/cards.names lists all 36 Baloot cards")

## 3. Load assets

Open the packed background and card shards via the loaders in `hakim_vision.synthetic.assets`. If this cell fails with `no shards found`, run the `pack-backgrounds` / `pack-cards` CLI commands from Section 1.

In [ ]:
from hakim_vision.synthetic import Backgrounds, Cards

BACKGROUNDS_DIR = Path("data/shards/backgrounds")
CARDS_DIR       = Path("data/shards/cards")

bg_shards   = sorted(BACKGROUNDS_DIR.glob("*.tar"))
card_shards = sorted(CARDS_DIR.glob("*.tar"))
assert bg_shards,   f"no background shards in {BACKGROUNDS_DIR} — see Section 1"
assert card_shards, f"no card shards in {CARDS_DIR} — see Section 1"

backgrounds = Backgrounds(bg_shards,   rng=np.random.default_rng(1))
cards       = Cards(card_shards,       rng=np.random.default_rng(2))

print(f"backgrounds  : {len(backgrounds)}")
print(f"card samples : {len(cards)}")
print(f"card classes : {len(cards.class_names)}")

shard_classes = set(cards.class_names)
assert shard_classes == declared, (
    f"packed shards do not match data/cards.names.\n"
    f"  only in shards     : {sorted(shard_classes - declared)}\n"
    f"  only in cards.names: {sorted(declared - shard_classes)}\n"
    f"Re-pack with the 36-card Baloot source directory."
)
print("✅ shards match the declared 36-card Baloot deck")

## 4. Render smoke scenes

Compose 8 random scenes with the package's `render_random_scene`. The RNG is seeded so two runs produce identical output.

In [ ]:
from hakim_vision.synthetic import render_random_scene

N_SCENES    = 8
CANVAS_SIZE = 512
rng = np.random.default_rng(0)

scenes = [
    render_random_scene(cards, backgrounds, rng=rng, n_cards=2, canvas_size=CANVAS_SIZE)
    for _ in range(N_SCENES)
]
label_counts = [len(s.labels) for s in scenes]
print(f"rendered {len(scenes)} scenes; labels per scene: {label_counts}")
print("✅ scenes rendered")

## 5. Visualize (eyeball check)

Draw the first 4 scenes with their VOC bounding boxes and class labels. Cards should look plausible, boxes should hug the rank/suit corners.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, scene in zip(axes.flat, scenes[:4]):
    rgb = cv2.cvtColor(scene.image, cv2.COLOR_BGR2RGB).copy()
    for label in scene.labels:
        xmin, ymin, xmax, ymax = label.voc
        cv2.rectangle(rgb, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
        cv2.putText(rgb, label.class_name, (xmin, max(ymin - 4, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    ax.imshow(rgb)
    ax.set_title(f"{len(scene.labels)} labels")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Label validity

Programmatic checks on every emitted label across all 8 scenes:

- class name is a known Baloot card
- class id is in `[0, 35]`
- YOLO `(cx, cy, w, h)` are all in `[0, 1]`
- VOC `(xmin, ymin, xmax, ymax)` are inside the canvas and strictly ordered

In [ ]:
class_to_id = {name: i for i, name in enumerate(cards.class_names)}
total_labels = 0

for scene_idx, scene in enumerate(scenes):
    for label in scene.labels:
        total_labels += 1
        assert label.class_name in class_to_id, f"unknown class {label.class_name!r}"
        cls_id = class_to_id[label.class_name]
        assert 0 <= cls_id <= 35, f"class id {cls_id} out of [0, 35]"
        assert 0.0 <= label.yolo.cx <= 1.0
        assert 0.0 <= label.yolo.cy <= 1.0
        assert 0.0 <  label.yolo.w  <= 1.0
        assert 0.0 <  label.yolo.h  <= 1.0
        xmin, ymin, xmax, ymax = label.voc
        assert 0 <= xmin < xmax <= CANVAS_SIZE
        assert 0 <= ymin < ymax <= CANVAS_SIZE

print(f"✅ validated {total_labels} labels across {len(scenes)} scenes")

## 7. Class-balance snapshot

Distribution of card classes across the smoke-test sample. **8 scenes is far too few for real balance** — this is just a sanity check that more than one class appeared. For statistical coverage on a production dataset, use:

```powershell
uv run hakim-vision generate --backgrounds data/shards/backgrounds --cards data/shards/cards -n 10000
```

In [ ]:
from collections import Counter

hist = Counter(label.class_name for scene in scenes for label in scene.labels)
for name in sorted(hist):
    print(f"  {name:>4}: {hist[name]}")
print(f"distinct classes seen: {len(hist)} / 36")

## 8. Tester sign-off

Tick each item before reporting the pipeline as green:

- [ ] All cells ran without exceptions
- [ ] Section 2 printed `✅ data/cards.names lists all 36 Baloot cards`
- [ ] Section 3 printed `✅ shards match the declared 36-card Baloot deck`
- [ ] Section 5 plots show recognisable playing cards on textured backgrounds, with green bounding boxes near the rank/suit corners
- [ ] Section 6 validated >0 labels with no assertion errors

If any item fails, file an issue with:
1. The failing cell's full traceback.
2. The output of `uv run hakim-vision config-show`.
3. Your OS, Python version, and the versions printed by Section 1.